In [142]:
import os
import pandas as pd

In [143]:
raw_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "raw", "MCDI_Overview_10-04-2025.xlsx")

df = pd.read_excel(raw_data_path, sheet_name="Parametric Analysis", engine="openpyxl", skiprows=2)

c:\Users\david\AppData\Local\pypoetry\Cache\virtualenvs\capbioco2-optimization-LdjI1cg_-py3.13\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\david\AppData\Local\pypoetry\Cache\virtualenvs\capbioco2-optimization-LdjI1cg_-py3.13\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [144]:
df.rename(columns={df.columns[0]: "Series", df.columns[1]: "Measurement"}, inplace=True)
df["Series"] = df["Series"].ffill()

def move_column(df, col_name, new_position):
    cols = list(df.columns)
    cols.insert(new_position, cols.pop(cols.index(col_name)))
    return df[cols]

def drop_columns_from(df, col_start):
    idx = list(df.columns).index(col_start)
    return df.iloc[:, :idx]

df = move_column(df, "Check", 2)
df = drop_columns_from(df, "Test Remarks ")


In [145]:
# Drop all measurements that are not valid
df = df[df["Check"] == "Valid"]

# Drop all measurements with Mol.\n(g/L) unequal to 0.5
df = df[df["Mol.\n(g/L)"] == 0.5]

# Drop all measurements with H2O unequal to DI
df = df[df["H2O\n"] == "DI"]

# Drop the 4th measurement series
df = df[df["Series"] != "4 -  pH Series"]

# Drop all measurements with a time in seconds unequal to 155/45/120
df = df[df["Time\n(s)"] == "155/45/120"]

In [ ]:
# Prepare input data

df_input = df[["Series", "Measurement", "Flow\n (L/min)", "Curr. Lim.\n(A)"]].copy()

df_input[["Flow_C", "Flow_PP", "Flow_P"]] = df_input["Flow\n (L/min)"].str.split("/", expand=True)
df_input.drop(columns="Flow\n (L/min)", inplace=True)

df_input["Current_P"] = df_input["Curr. Lim.\n(A)"].str.split("/", expand=True)[2]
df_input.drop(columns="Curr. Lim.\n(A)", inplace=True)

In [151]:
# Prepate output data

df_output = df[["Series", "Measurement", "Remov. \n(%)", "Ry \n(%)"]].copy()
df_output.rename(columns={"Remov. \n(%)": "Removal", "Ry \n(%)": "Recovery"}, inplace=True)

In [152]:
df_output

,Series,Measurement,Removal,Recovery
10,"1 - Pre-Optimization 0,5",1.1,96.0,36.0
11,"1 - Pre-Optimization 0,5",1.2,95.0,36.0
12,"1 - Pre-Optimization 0,5",1.3,89.0,34.0
13,"1 - Pre-Optimization 0,5",1.4,83.0,33.0
14,"1 - Pre-Optimization 0,5",1.5,97.0,36.0
19,2 - Current/Flow-Field 0.5 g/l,2.2,98.0,34.0
20,2 - Current/Flow-Field 0.5 g/l,2.3,98.0,33.0
21,2 - Current/Flow-Field 0.5 g/l,2.4,98.0,34.0
22,2 - Current/Flow-Field 0.5 g/l,2.5,98.0,32.0
23,2 - Current/Flow-Field 0.5 g/l,2.6,98.0,33.0
